# CBIS-DDSM: **Attention-MIL** Weakly-Supervised Localization

The classifier+Grad-CAM approach failed (pointing-game ~0-2%) because mass/calc is a *global*
task, so the CAM never focuses on the lesion. This uses the **proper** weakly-supervised method:

- Each mammogram = a **bag** of **4x4 = 16 tiles**.
- A frozen EfficientNetB0 encodes every tile (features precomputed once -> ~230 MB, RAM-safe).
- A **gated-attention MIL pooling** head (Ilse et al. 2018) learns a per-tile attention weight and
  classifies the bag (mass vs calc). To tell mass from calc, attention *must* land on the lesion
  tile (normal tiles are identical across classes) -> the attention **is** the localization.

**Localization eval:** the 4x4 attention grid vs tile-level lesion ground truth (a tile is
positive if the ROI mask overlaps it): **pointing-game** (top-attention tile is a lesion tile),
**tile-AUC** (attention ranks lesion tiles high), and attention-vs-lesion **IoU**.


In [ ]:
# ---- GPU: pin to RTX (hide Intel iGPU) ----
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if gpus:
    try:
        tf.config.set_visible_devices([gpus[0]], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e:
        print('gpu cfg:', e)
print('Visible:', tf.config.get_visible_devices('GPU'))

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# ============================================================ CONFIG
DATA_ROOT  = 'D:/cbis-ddsm'; JPEG_ROOT = DATA_ROOT + '/jpeg'; CSV_DIR = DATA_ROOT + '/csv'
DICOM_INFO = CSV_DIR + '/dicom_info.csv'; SAVE_DIR = 'G:/My Drive/VS_CC/models'
os.makedirs(SAVE_DIR, exist_ok=True)
TRAIN_CSVS = ['mass_case_description_train_set.csv', 'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',  'calc_case_description_test_set.csv']
TASK = 'abnormality'          # mass vs calc
GRID = 4                       # 4x4 = 16 tiles per bag
TILE = 224
K = GRID * GRID
FEAT_DIM = 1280               # EfficientNetB0 GAP
EPOCHS = 60; BATCH = 16; RANDOM_STATE = 2021; MASK_THRESH = 127

## 1. Resolve full image + ROI mask (via dicom_info.csv)

In [ ]:
def _to_local(p):
    p = str(p).replace('\\', '/'); i = p.find('jpeg/')
    return os.path.join(JPEG_ROOT, p[i + 5:]) if i >= 0 else p
def build_uid_maps(dinfo):
    di = pd.read_csv(dinfo); di['SeriesDescription'] = di['SeriesDescription'].fillna('')
    di['local'] = di['image_path'].apply(_to_local); di['uid'] = di['SeriesInstanceUID'].astype(str)
    full, mask, allf = {}, {}, {}
    for _, r in di.iterrows():
        allf.setdefault(r['uid'], []).append(r['local']); d = r['SeriesDescription'].lower()
        if 'full' in d: full.setdefault(r['uid'], r['local'])
        elif 'roi mask' in d: mask.setdefault(r['uid'], r['local'])
    return full, mask, allf
FULL_MAP, MASK_MAP, ANY_MAP = build_uid_maps(DICOM_INFO)
def series_uid(p):
    q = str(p).replace('\\', '/').strip().split('/'); return q[-2] if len(q) >= 2 else None
def is_mask_like(g):
    return g is not None and np.mean((g <= 10) | (g >= 245)) > 0.90 and len(np.unique(g)) <= 64
def resolve_mask(p):
    u = series_uid(p)
    if u in MASK_MAP: return MASK_MAP[u]
    for c in ANY_MAP.get(u, []):
        if is_mask_like(cv2.imread(c, cv2.IMREAD_GRAYSCALE)): return c
    return None
def load_tables(names):
    df = pd.concat([pd.read_csv(os.path.join(CSV_DIR, n)) for n in names if os.path.exists(os.path.join(CSV_DIR, n))], ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]; return df
def prep(df):
    src = df['abnormality_type'] if 'abnormality_type' in df.columns else df['abnormality']
    df['label'] = src.astype(str).str.upper().str.contains('MASS').astype(int)
    df['full_path'] = df['image_file_path'].apply(lambda p: FULL_MAP.get(series_uid(p)))
    df['mask_path'] = df['roi_mask_file_path'].apply(resolve_mask)
    df['patient_id'] = df['patient_id'].astype(str); df['image_key'] = df['image_file_path'].astype(str)
    return df[['patient_id', 'image_key', 'full_path', 'mask_path', 'label']].dropna(subset=['full_path'])
train_df = prep(load_tables(TRAIN_CSVS)); test_df = prep(load_tables(TEST_CSVS))
ov = set(train_df.patient_id) & set(test_df.patient_id)
if ov: train_df = train_df[~train_df.patient_id.isin(ov)].reset_index(drop=True)
print('Train', len(train_df), 'Test', len(test_df), 'disjoint', set(train_df.patient_id).isdisjoint(set(test_df.patient_id)))

## 2. Tiling + tile-level ground truth from ROI masks

In [ ]:
_clahe = cv2.createCLAHE(2.0, (8, 8))
CANVAS = GRID * TILE
def tile_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img = _clahe.apply(cv2.resize(img, (CANVAS, CANVAS)))
    tiles = np.empty((K, TILE, TILE, 3), dtype=np.uint8); n = 0
    for i in range(GRID):
        for j in range(GRID):
            tiles[n] = np.stack([img[i*TILE:(i+1)*TILE, j*TILE:(j+1)*TILE]] * 3, -1); n += 1
    return tiles
def mask_tile_grid(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) if path else None
    if not is_mask_like(m): return None
    m = (cv2.resize(m, (CANVAS, CANVAS), interpolation=cv2.INTER_NEAREST) >= MASK_THRESH).astype(np.uint8)
    g = m.reshape(GRID, TILE, GRID, TILE).sum(axis=(1, 3))     # lesion pixels per tile
    return (g > 0).astype(int)                                  # (GRID,GRID) tile-positive

## 3. Precompute frozen tile features (one pass, ~230 MB)

In [ ]:
_b = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(TILE, TILE, 3))
feat_model = Model(_b.input, layers.GlobalAveragePooling2D()(_b.output))

def extract(df, want_grid):
    F = np.empty((len(df), K, FEAT_DIM), dtype=np.float32); Y = np.empty(len(df), dtype=np.int64)
    keys, grids, j = [], [], 0
    for row in df.itertuples(index=False):
        tiles = tile_image(row.full_path)
        if tiles is None: continue
        F[j] = feat_model.predict(tiles, verbose=0)
        Y[j] = row.label; keys.append(row.image_key)
        if want_grid:
            grids.append(mask_tile_grid(row.mask_path) if row.mask_path else None)
        j += 1
        if j % 400 == 0: print('  ...', j)
    return F[:j], Y[:j], keys, grids

print('Extracting TRAIN tile features...'); Ftr, ytr, ktr, _ = extract(train_df, False)
print('Extracting TEST tile features...');  Fte, yte, kte, grids_te = extract(test_df, True)
print('Ftr', Ftr.shape, 'Fte', Fte.shape, '| approx MB', round((Ftr.nbytes+Fte.nbytes)/1e6))

## 4. Gated-attention MIL head

In [ ]:
def build_mil(feat_dim=FEAT_DIM, k=K, hid=128, att=64, lr=1e-3):
    inp = layers.Input((k, feat_dim))
    h = layers.TimeDistributed(layers.Dense(hid, activation='relu'))(inp)
    h = layers.TimeDistributed(layers.Dropout(0.3))(h)
    V = layers.TimeDistributed(layers.Dense(att, activation='tanh'))(h)
    U = layers.TimeDistributed(layers.Dense(att, activation='sigmoid'))(h)
    s = layers.TimeDistributed(layers.Dense(1))(layers.Multiply()([V, U]))     # (k,1)
    s = layers.Flatten()(s)
    a = layers.Softmax(name='attention')(s)                                    # (k,)
    bag = layers.Dot(axes=1)([a, h])                                           # (hid,)
    out = layers.Dense(1, activation='sigmoid')(bag)
    m = Model(inp, out)
    m.compile(Adam(lr), 'binary_crossentropy', ['accuracy', tf.keras.metrics.AUC(name='auc')])
    amodel = Model(inp, m.get_layer('attention').output)
    return m, amodel

mil, attn_model = build_mil()
mil.summary(line_length=90)

## 5. Train the MIL head (fast; features are fixed)

In [ ]:
gss = GroupShuffleSplit(1, test_size=0.15, random_state=RANDOM_STATE)
tr, va = next(gss.split(Ftr, ytr, groups=np.array(ktr)))
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=ytr[tr])
mil.fit(Ftr[tr], ytr[tr], validation_data=(Ftr[va], ytr[va]), epochs=EPOCHS, batch_size=BATCH,
        class_weight={0: cw[0], 1: cw[1]},
        callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True)], verbose=2)

## 6. Bag classification on the patient-disjoint test set

In [ ]:
prob = mil.predict(Fte, verbose=0).ravel(); pred = (prob >= 0.5).astype(int)
def auc_ci(yt, yp, n=2000, seed=42):
    rng = np.random.RandomState(seed); s = []
    for _ in range(n):
        idx = rng.randint(0, len(yt), len(yt))
        if len(np.unique(yt[idx])) == 2: s.append(roc_auc_score(yt[idx], yp[idx]))
    return np.percentile(s, [2.5, 97.5])
auc = roc_auc_score(yte, prob); lo, hi = auc_ci(yte, prob)
print(f'Bag Accuracy {accuracy_score(yte, pred):.3f} | ROC-AUC {auc:.3f} (95% CI {lo:.3f}-{hi:.3f})')
print(classification_report(yte, pred, target_names=['Calc', 'Mass']))

## 7. Weakly-supervised localization: attention grid vs tile-level GT

In [ ]:
A = attn_model.predict(Fte, verbose=0)               # (N, K) attention weights
pg, ious, tile_aucs = [], [], []
for a_vec, grid in zip(A, grids_te):
    if grid is None or grid.sum() == 0: continue
    a_grid = a_vec.reshape(GRID, GRID)
    gt = grid.astype(int)
    # pointing game: top-attention tile is a lesion tile
    top = np.unravel_index(int(np.argmax(a_grid)), a_grid.shape)
    pg.append(int(gt[top] == 1))
    # tile-AUC: does attention rank lesion tiles above normal tiles?
    if len(np.unique(gt)) == 2:
        tile_aucs.append(roc_auc_score(gt.ravel(), a_vec))
    # IoU of top-attention tiles (>= half of max) vs GT tiles
    pm = (a_grid >= 0.5 * a_grid.max()).astype(int)
    inter = (pm & gt).sum(); union = (pm | gt).sum()
    ious.append(inter / union if union else 0.0)
print(f'n={len(pg)}  pointing-game={np.mean(pg):.3f}  tile-AUC={np.mean(tile_aucs):.3f}  IoU={np.mean(ious):.3f}')
loc = dict(n=len(pg), pointing_game=float(np.mean(pg)), tile_auc=float(np.mean(tile_aucs)), iou=float(np.mean(ious)))

### 7b. Overlays: attention grid + GT lesion tiles

In [ ]:
show = [i for i, g in enumerate(grids_te) if g is not None and g.sum() > 0][:6]
fig, ax = plt.subplots(1, len(show), figsize=(3*len(show), 3.2), dpi=110); ax = np.atleast_1d(ax)
for a, idx in zip(ax, show):
    row = test_df[test_df['image_key'] == kte[idx]].iloc[0]
    img = cv2.resize(cv2.imread(row['full_path'], cv2.IMREAD_GRAYSCALE), (CANVAS, CANVAS))
    ag = A[idx].reshape(GRID, GRID)
    heat = cv2.resize(ag / (ag.max()+1e-8), (CANVAS, CANVAS), interpolation=cv2.INTER_NEAREST)
    a.imshow(img, cmap='gray'); a.imshow(heat, cmap='jet', alpha=0.4)
    gt = grids_te[idx]
    for i in range(GRID):
        for j in range(GRID):
            if gt[i, j]: a.add_patch(plt.Rectangle((j*TILE, i*TILE), TILE, TILE, ec='lime', fc='none', lw=1.5))
    a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'cbis_mil_overlays.png'), dpi=200, bbox_inches='tight'); plt.show()

## 8. Summary + save

In [ ]:
summary = (
    f"Attention-MIL (gated attention over a {GRID}x{GRID}-tile bag, frozen EfficientNetB0 tile "
    f"encoder) on CBIS-DDSM full mammograms for mass vs. calcification (patient-disjoint split). "
    f"Bag ROC-AUC {auc:.3f} (95% CI {lo:.3f}-{hi:.3f}). Weakly-supervised localization from the "
    f"attention weights, scored against tile-level ROI-mask ground truth over {loc['n']} masked "
    f"test mammograms: pointing-game {loc['pointing_game']*100:.1f}%, tile-AUC {loc['tile_auc']:.3f}, "
    f"IoU {loc['iou']:.3f}. (Grad-CAM baselines were pointing-game ~0-2%.)"
)
print(summary)
open(os.path.join(SAVE_DIR, 'cbis_mil_summary.txt'), 'w').write(summary)
mil.save_weights(os.path.join(SAVE_DIR, 'cbis_mil.weights.h5'))
print('\nSaved to', SAVE_DIR)